# MicroDuck 官方展示
这份 Notebook **只展示 Pollen Robotics 官方 MicroDuck**：官方仓库做了什么、有哪些强化学习任务、官方策略实际跑起来什么样。不含任何大模型。

1. **介绍**：下面的文字不用运行就能看。
2. **试驾**：运行第 1、2 格后，第 3 格启动遥控台，用方向键驾驶官方 ONNX 步态，同屏看第三视角、机器人第一视角和俯视轨迹。机器人和步态都是官方的，2×2 房间场景由本仓库搭建。
3. **任务与策略**：第 4、5 格列出全部强化学习任务、奖励和训练配置（只需运行第 1 格）；第 6 格在仿真里实跑 10 个官方策略（需要第 1、2 格）。

默认申请 GPU 运行时，只用来渲染画面；没有 GPU 会自动改用 CPU 软件渲染，只是更慢。

**想保留运行结果**：运行前先点 **Copy to Drive**（或 文件 → 在云端硬盘中保存一份副本），在副本里运行。第 4–6 格的表格和动画会存进 ipynb，换一台电脑用同一账号打开副本，不连接运行时也能看到。第 3 格遥控台是实时界面，不会保存。

## MicroDuck 原始仓库做了什么

MicroDuck 是 Pollen Robotics 的双足小机器人，约 25 cm 高、800 g。官方工作分三处，按“训练 → 发布 → 上机”衔接：

| 仓库 | 作用 | 本 Notebook 用到的部分 |
|---|---|---|
| [microduck_rl](https://github.com/pollen-robotics/microduck_rl) | 在 MuJoCo 里用强化学习训练动作 | 机器人模型、场景、任务定义 |
| [microduck-policies](https://huggingface.co/pollen-robotics/microduck-policies) | 发布训练好的 ONNX 策略 | 全部 10 个策略 |
| [microduck](https://github.com/pollen-robotics/microduck) | 真机上的运行时（Rust） | 不运行，只介绍 |

### 1 · microduck_rl：强化学习训练

- **训练框架**：[mjlab](https://github.com/mujocolab/mjlab)（MuJoCo Warp）+ PPO，官方命令用 4096 个并行环境，需要 CUDA GPU。控制频率 50 Hz。
- **统一接口**：所有任务共用 61 维观测（48 维本体感觉 + 13 维指令）和 14 维动作（14 个舵机的目标位置），所以真机可以随时热切换策略。
- **缩小仿真与真机差距**：用 [BAM](https://github.com/Rhoban/bam) 模拟 XL330 舵机的电压控制和摩擦；随机化电压、延迟、摩擦、质心、质量、IMU 安装误差和编码器偏差；另有模拟齿轮间隙的 Backlash 任务变体。
- **导出**：把观测归一化写进 ONNX，再发布到 Hugging Face。

共注册 33 个任务：18 个基础任务和 15 个 Backlash 变体。基础任务分成 13 个任务族：

| 类别 | 任务族 |
|---|---|
| 走与站 | 速度行走 Velocity、走站起一体 VelStand、起身站立 StandUp、坐下起立 SitStand |
| 动作 | 低头啄地 GroundPick、踢球 BallKick、前滚翻 Roulade |
| 轮滑 | 轮滑 Rollers、葫芦步 Swizzle、下蹲滑行 RollerCrouch、下坡 RollerSlope、轮滑起身 RollerStandUp、原地旋转 Spin |

### 2 · microduck-policies：官方策略与训练检查点

| ONNX 文件 | 动作 | 类型 |
|---|---|---|
| `velstand.onnx` | 走、站、倒地起身（真机默认步态） | 持续 |
| `alpha_walking.onnx` | 行走（第 3 格遥控台用的就是它） | 持续 |
| `alpha_stand.onnx` | 起身、站立、身体姿态 | 持续 |
| `alpha_sitstand.onnx` | 坐下 / 起立 | 由姿势标志控制 |
| `alpha_ground_pick.onnx` | 低头啄地 | 一次性，相位驱动 |
| `ball_kick_left.onnx`、`ball_kick_right.onnx` | 左 / 右脚踢球 | 一次性 |
| `roulade.onnx` | 前滚翻 | 一次性，可连滚 |
| `roller.onnx` | 轮滑 | 持续，轮滑模式 |
| `roller_crouch.onnx` | 轮滑下蹲滑行 | 一次性，相位驱动，轮滑模式 |

Swizzle、RollerSlope、RollerStandUp、Spin 四个任务族有训练代码，但官方策略集里没有标明对应的权重。

ONNX 只能运行。想在官方成果上**继续训练**，要用 [microduck-training](https://huggingface.co/pollen-robotics/microduck-training)：它公开了 9 份完整检查点，覆盖行走、走站起、起身、坐站、啄地、左右踢球、前滚翻和轮滑下蹲。

### 3 · microduck：真机运行时

跑在 Rockchip RK3566 上的一组 Rust 守护进程，之间用 Unix socket 上的 JSON-RPC 2.0 通信。手机、手柄、命令行发出的都是“意图”，只有 `robotd` 能驱动电机，并由它判断是否安全。

| 进程 | 职责 |
|---|---|
| `robotd` | 50 Hz 控制循环：读 15 个舵机（14 个策略控制 + 嘴）和 IMU、运行策略、跌倒检测、安全限位 |
| `updaterd` | 签名更新：验签、整目录切换、健康检查，不健康就自动回滚 |
| `configd` | Wi-Fi、机器人身份、配对 PIN、手柄绑定、重启 |
| `btd` | 手机蓝牙入口 |
| `padd` / `netpadd` | 蓝牙或 USB 手柄 / 局域网 UDP 手柄 |
| `mediad` | 摄像头和麦克风、WebRTC 视频、远程访问入口 |
| `tofd` | 头部 8×8 深度传感器 |
| `mapd` | 机载地图与定位 |
| `autod` | 自主行为：闲逛、张望、打盹、避障（默认关闭） |
| `nfcd` | NFC 标签：碰一下头部即可配对手柄或切换行走、轮滑、激光、足球、自主模式 |

主要功能：

- **手柄**：A 坐下 / 起立，B 啄地，LB / RB 是点头、摇头（足球模式下为左右脚踢球），方向键切换摇杆控制的对象。
- **技能与策略**：`robotctl policy` 可从 Hugging Face 安装、替换、绑定策略，不用升级系统。
- **玩法**：追激光点、自主踢紫色球、13 种表情动作、多鸭合唱、用深度传感器当特雷门琴。
- **保护与记录**：被抱起时自动暂停步态；`vetbook` 记录机器人一生的数据。
- **仿真替身**：`scripts/duck-sim` 让同一套守护进程驱动 MuJoCo 里的鸭子，代码与真机一致。

周边仓库：[microduck_yocto](https://github.com/pollen-robotics/microduck_yocto)（操作系统镜像，A/B 分区更新）、[microduck-arena](https://github.com/pollen-robotics/microduck-arena) 与 [microduck-challenges](https://github.com/pollen-robotics/microduck-challenges)（策略竞赛与训练模板）、[duck_detector](https://github.com/pollen-robotics/duck_detector)（识别其他鸭子）、[microduck_emotions](https://github.com/pollen-robotics/microduck_emotions)（表情动作）。

> 以上内容整理自 microduck_rl `0daab16`、microduck-policies v10（`236d183`）和 microduck `4024f6e`（2026-11-25 查阅）。官方仓库仍在更新，以原仓库为准。

In [ ]:
# @title 1 · 安装原始 MicroDuck 仿真环境
project_source = "github"  # @param ["github", "drive", "upload"]
drive_folder = "microduck-qwen"  # @param {type:"string"}
render_backend = "auto"  # @param ["auto", "egl", "osmesa"]

import os
import shutil
import subprocess
import sys
import tempfile
import tomllib
import zipfile
from pathlib import Path

if sys.version_info[:2] not in ((3, 12), (3, 13)):
    raise RuntimeError("请选择 Python 3.12 或 3.13 的运行时，然后重新运行。")

if render_backend == "auto":  # 有 NVIDIA GPU 就用 GPU 渲染，否则退回 CPU 软件渲染
    render_backend = "egl" if Path("/dev/nvidia0").exists() else "osmesa"

root = Path("/content/microduck-qwen")
root.mkdir(parents=True, exist_ok=True)
bundle = "microduck-qwen-colab-source.zip"
local_zip = Path("/content") / bundle

if project_source == "github":
    with tempfile.TemporaryDirectory() as update:
        staged = Path(update) / "source"
        subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
                        "https://github.com/atlasmarkz/microduck-qwen.git", str(staged)], check=True)
        subprocess.run(["git", "-C", str(staged), "sparse-checkout", "set", "microduck_qwen"], check=True)
        actual = subprocess.check_output(["git", "-C", str(staged), "rev-parse", "--short", "HEAD"], text=True).strip()
        shutil.copytree(staged / "microduck_qwen", root / "microduck_qwen", dirs_exist_ok=True)
        shutil.copyfile(staged / "pyproject.toml", root / "pyproject.toml")
        print("已从 GitHub 同步源码，提交：", actual)
elif project_source == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    drive_dir = Path("/content/drive/MyDrive") / drive_folder
    drive_zip = drive_dir / bundle
    if drive_zip.is_file():
        shutil.copyfile(drive_zip, local_zip)
        with zipfile.ZipFile(local_zip) as archive:
            archive.extractall(root)
        print("已从 Google Drive 解压最新源码包：", drive_zip)
    elif (drive_dir / "pyproject.toml").is_file() and (drive_dir / "microduck_qwen").is_dir():
        shutil.copytree(drive_dir / "microduck_qwen", root / "microduck_qwen", dirs_exist_ok=True)
        shutil.copyfile(drive_dir / "pyproject.toml", root / "pyproject.toml")
        print("已从 Google Drive 项目目录同步源码：", drive_dir)
    else:
        raise FileNotFoundError(f"未在 {drive_dir} 找到 {bundle} 或项目源码，也可切换到 github 或 upload。")
else:
    from google.colab import files
    uploaded = files.upload()
    if bundle not in uploaded:
        raise FileNotFoundError(f"请选择 {bundle}。")
    local_zip.write_bytes(uploaded[bundle])
    with zipfile.ZipFile(local_zip) as archive:
        archive.extractall(root)

os.environ["MUJOCO_GL"] = render_backend
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
for _mod in [m for m in list(sys.modules) if m == "microduck_qwen" or m.startswith("microduck_qwen.")]:
    del sys.modules[_mod]

def run(*cmd):
    completed = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    if completed.returncode:
        raise RuntimeError(completed.stdout[-8500:])

deps_marker = Path(f"/content/.microduck_deps_cpu_{render_backend}")
if deps_marker.is_file():
    print("当前会话依赖已安装，跳过重复安装。")
else:
    gl_packages = ("libegl1", "libgl1") if render_backend == "egl" else ("libosmesa6", "libgl1")
    subprocess.run(["apt-get", "-qq", "update"], check=True)
    run("apt-get", "-qq", "-y", "install", *gl_packages)
    project = tomllib.loads((root / "pyproject.toml").read_text())["project"]
    requirements = project["optional-dependencies"]
    bam = Path("/content/microduck-bam.zip")
    bam_url = next(item.split(" @ ", 1)[1] for item in requirements["simulation"] if " @ " in item)
    run("curl", "-fsSL", "--retry", "3", bam_url, "-o", str(bam))
    sim_packages = [item for item in requirements["simulation"] if " @ " not in item]
    with tempfile.TemporaryDirectory() as build:
        bam_source = bam
        if sys.version_info[:2] == (3, 13):
            with zipfile.ZipFile(bam) as archive:
                archive.extractall(build)
            manifest, = Path(build).glob("bam-*/pyproject.toml")
            config = manifest.read_text(encoding="utf-8")
            guard = 'requires-python = ">=3.12,<3.13"'
            if config.count(guard) != 1:
                raise RuntimeError("固定版 BAM 的 Python 声明与预期不符。")
            manifest.write_text(config.replace(guard, 'requires-python = ">=3.12,<3.14"'),
                                encoding="utf-8")
            bam_source = manifest.parent
        run(sys.executable, "-m", "pip", "install", "-q", "huggingface-hub", "Pillow>=10.1", "ipywidgets",
            *sim_packages, str(bam_source))
    deps_marker.write_text("ok")
from microduck_qwen.colab_demo import prepare_rendering
print("渲染器：", prepare_rendering(render_backend))
print("依赖安装完成。")

In [ ]:
# @title 2 · 下载官方 ONNX 步态和仿真资产
import tarfile
import tempfile
from pathlib import Path
from huggingface_hub import hf_hub_download
from microduck_qwen.sim_native import DEFAULT_POLICY, DEFAULT_RL, POLICY_REVISION
hf_hub_download("pollen-robotics/microduck-policies", "alpha_walking.onnx",
                revision=POLICY_REVISION, cache_dir=DEFAULT_POLICY.parents[3], token=False)

upstream_rev = "0daab160d3fb1c2cbaba16a4a19052f36709dd9e"
if not DEFAULT_RL.exists():
    DEFAULT_RL.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(dir=DEFAULT_RL.parent) as temporary:
        upstream_archive = Path(temporary) / "upstream.tar.gz"
        run("curl", "-fsSL", "--retry", "3",
            f"https://codeload.github.com/pollen-robotics/microduck_rl/tar.gz/{upstream_rev}",
            "-o", str(upstream_archive))
        with tarfile.open(upstream_archive) as archive:
            archive.extractall(temporary, filter="data")
        (Path(temporary) / f"microduck_rl-{upstream_rev}").rename(DEFAULT_RL)

assert DEFAULT_POLICY.is_file(), "官方步态下载不完整。"
print("官方步态和仿真资产就绪。")

In [ ]:
# @title 3 · 试驾官方步态：交互式遥控台
seed = 98107  # @param {type:"integer"}

from microduck_qwen.colab_demo import interactive_baseline
console = interactive_baseline(seed=seed)

## 原始 MicroDuck 的强化学习任务与官方策略

下面三格与遥控台互相独立：第 4、5 格只需先运行第 1 格，第 6 格需要第 1、2 格。任务数据是从固定提交的真实任务注册表导出的快照，不是手抄。

In [ ]:
# @title 4 · 全部 RL 任务总表
from microduck_qwen.upstream_tour import show_tasks
show_tasks()

In [ ]:
# @title 5 · 查看单个任务：观测、奖励、课程、PPO 配置
# @markdown 换一个任务后重新运行本格。
task = "Velocity · 速度行走"  # @param ["Velocity · 速度行走", "VelStand · 走站起一体", "StandUp · 起身站立", "SitStand · 坐下起立", "GroundPick · 低头啄地", "BallKick · 踢球", "Roulade · 前滚翻", "Rollers · 轮滑", "Swizzle · 轮滑葫芦步", "RollerCrouch · 轮滑下蹲滑行", "RollerSlope · 轮滑下坡", "RollerStandUp · 轮滑起身", "Spin · 轮滑原地旋转"]

from microduck_qwen.upstream_tour import show_family
show_family(task)

In [ ]:
# @title 6 · 官方策略技能展厅：在仿真里实跑并播放
# @markdown 选一个技能后运行本格；换技能就再运行一次。CPU 软件渲染时单个技能约需一两分钟，GPU 渲染会快很多。<br>
# @markdown 动画左上角是当前控制机器人的策略，黄色表示正在演示的策略。
skill = "roulade · 前滚翻"  # @param ["walk · 行走", "velstand · 走站一体（含倒地起身）", "stand · 站立与身体姿态", "sitstand · 坐下 / 起立", "ground_pick · 低头啄地", "kick_left · 左脚踢球", "kick_right · 右脚踢球", "roulade · 前滚翻", "roller · 轮滑", "roller_crouch · 轮滑下蹲滑行", "all · 全部依次运行"]

from microduck_qwen.upstream_tour import show_skill
show_skill(skill)

## 接下来

官方策略接收的是速度、姿态这类低层指令。[MicroDuck × Qwen-VL](https://colab.research.google.com/github/atlasmarkz/microduck-qwen/blob/main/notebooks/microduck-live-demo.ipynb) 展示本仓库在官方行走步态之上做的工作：用中文指令，让模型看图决策并完成跨房间任务。